# 1 · A warehouse robot graph in LanceDB

This is the first of three notebooks. In this one we build a property graph,
store it in LanceDB, and query it with Cypher through lance-csr. There's no
hyperdimensional computing (HDC) yet, because we're starting here with the graph — it will be our
**source of truth** later on.

The general idea we showcase in these three notebooks is the following:

> **HDC proposes candidates. The graph validates whether those paths actually exist.**

## The warehouse scenario

Our example scenario involved three robots working in a warehouse that has three aisles.
- **R17** had a near-miss with a forklift
in aisle 8.
- **R04** turned away from aisle 8, then skidded on oil in aisle 9 and found that
slowing down fixed it.
- **R11** went into aisle 9 and already knew to slow down.

Aisle 7 is blocked by a fallen pallet, and nobody visits it.

This scenario can be modeled as a graph:

```mermaid
graph LR
    R17((R17)) -- VISITED --> A8[aisle-8]
    R04((R04)) -- VISITED --> A8
    R04 -- VISITED --> A9[aisle-9]
    R11((R11)) -- VISITED --> A9
    A7[aisle-7] -- HAS_HAZARD --> P{{pallet}}
    A8 -- HAS_HAZARD --> F{{forklift}}
    A9 -- HAS_HAZARD --> O{{oil}}
    F -- HANDLED_BY --> RR([reroute])
    O -- HANDLED_BY --> SD([slow_down])
```

We have a simple graph with 11 nodes with 4 labels, and 9 edges with 3 relationship types. We'll use LanceDB and [lance-csr](https://github.com/hyperdimensionalcomputingai/lance-csr) to store this graph and then run Cypher queries over the data.

## Setup

LanceDB's Rust core logs a `WARN ... No existing dataset` line the first time it creates a
table. That's expected, not an error, so we turn the log level down **before** importing
`lancedb`.

In [1]:
import os
os.environ["LANCEDB_LOG"] = "error"  # must be set before lancedb is imported

import lancedb
from lance_csr import CypherQuery, GraphConfig

## The data

A labeled property graph (LPG) defines two key things: **nodes** that are grouped by node labels, and **edges**, grouped by
relationship labels. Here, both are plain Python literals. In the raw data, each node record is available as a dict with an `id`,
and each edge is a `(src, dst)` pair of node IDs.

In [2]:
NODES = {
    "Robot":    [{"id": "R17"}, {"id": "R04"}, {"id": "R11"}],
    "Aisle":    [{"id": "aisle-7"}, {"id": "aisle-8"}, {"id": "aisle-9"}],
    "Hazard":   [{"id": "pallet"}, {"id": "forklift"}, {"id": "oil"}],
    "Response": [{"id": "reroute"}, {"id": "slow_down"}],
}

In [3]:
EDGES = {
    "VISITED":    [("R17", "aisle-8"), ("R04", "aisle-8"), ("R04", "aisle-9"), ("R11", "aisle-9")],
    "HAS_HAZARD": [("aisle-7", "pallet"), ("aisle-8", "forklift"), ("aisle-9", "oil")],
    "HANDLED_BY": [("forklift", "reroute"), ("oil", "slow_down")],
}
print(sum(map(len, NODES.values())), "nodes,", sum(map(len, EDGES.values())), "edges")

11 nodes, 9 edges


## Store it in LanceDB

[LanceDB](https://docs.lancedb.com/) is an embedded vector retrieval library: in open source, it's a directory on disk, with no server to run. We store
the graph the way a relational engine would do, using the concept of **tables**.

Each node label gets its own table with an
`id` column, and each relationship type gets its own table with `src` and `dst` columns.
`mode="overwrite"` means you can re-run this cell as often as you like.

In [4]:
db = lancedb.connect("warehouse.lancedb")

for label, rows in NODES.items():
    db.create_table(label, rows, mode="overwrite")

for rel, pairs in EDGES.items():
    db.create_table(rel, [{"src": s, "dst": d} for s, d in pairs], mode="overwrite")

db.list_tables().tables

['Aisle', 'HANDLED_BY', 'HAS_HAZARD', 'Hazard', 'Response', 'Robot', 'VISITED']

## Read it back as Arrow

The Lance format uses Apache Arrow's type system. As a result, every LanceDB table reads straight into a PyArrow table, the columnar in-memory
format. lance-csr runs its queries directly on these Arrow tables, so there's no export or
import step between storage and the graph engine. Note that a relationship table is just an
**edge list** stored separately from the node table, with identifiers to the source/target node IDs.

In [5]:
db.open_table("VISITED").to_arrow()

pyarrow.Table
src: string
dst: string
----
src: [["R17","R04","R04","R11"]]
dst: [["aisle-8","aisle-8","aisle-9","aisle-9"]]

In [6]:
datasets = {name: db.open_table(name).to_arrow() for name in (*NODES, *EDGES)}
{name: t.num_rows for name, t in datasets.items()}

{'Robot': 3,
 'Aisle': 3,
 'Hazard': 3,
 'Response': 2,
 'VISITED': 4,
 'HAS_HAZARD': 3,
 'HANDLED_BY': 2}

## Describe the graph's shape

The tables alone don't say which ones are nodes and which are edges. A `GraphConfig` tells
lance-csr: for each node label, the column that holds the node ID; for each relationship
type, its source and target node tables and the columns that hold their IDs. It plays the same role as a schema
in a graph database.

In [7]:
builder = GraphConfig.builder()
for label in NODES:
    builder = builder.node_table(label, "id")
builder = (
    builder.rel_table("VISITED", "Robot", "Aisle", "src", "dst")
    .rel_table("HAS_HAZARD", "Aisle", "Hazard", "src", "dst")
    .rel_table("HANDLED_BY", "Hazard", "Response", "src", "dst")
)

cfg = builder.build()
cfg

GraphConfig(nodes=["aisle", "response", "robot", "hazard"], relationships=["has_hazard", "handled_by", "visited"])

## A first Cypher query: one hop

"Which aisles did R04 visit?" The pattern `(r:Robot {id: 'R04'})-[:VISITED]->(a:Aisle)`
follows one `VISITED` edge out of R04. The result is an Arrow table, which we turn into a
pandas DataFrame to show it.

In [8]:
q = "MATCH (r:Robot {id: 'R04'})-[:VISITED]->(a:Aisle) RETURN a.id"
CypherQuery(q).with_config(cfg).execute(datasets).to_pandas()

,a.id
0,aisle-8
1,aisle-9


## What lance-csr does under the hood

There's no separate graph store here. In this Arrow-based example, lance-csr compiles
the Cypher pattern into joins that Apache DataFusion runs over the tables. Here is the
SQL for the query above. lance-csr also offers an optional CSR adjacency index through
`CypherEngine`; this small example uses the default query path.

In [9]:
print(CypherQuery(q).with_config(cfg).to_sql(datasets))

SELECT a__id AS "a.id" FROM (SELECT visited_1__dst FROM (SELECT robot.id AS r__id FROM robot WHERE (robot.id = 'R04')) INNER JOIN (SELECT visited.src AS visited_1__src, visited.dst AS visited_1__dst FROM visited) ON r__id = visited_1__src) INNER JOIN (SELECT aisle.id AS a__id FROM aisle) ON visited_1__dst = a__id


## A multi-hop query that tells the story

"R11 is heading into aisle 9. What hazard might it meet, and how was that hazard handled?"
This takes three hops, `VISITED → HAS_HAZARD → HANDLED_BY`, and a graph database answers it
exactly.

In [10]:
q = '''
MATCH (r:Robot {id: 'R11'})-[:VISITED]->(a:Aisle)-[:HAS_HAZARD]->(h:Hazard)-[:HANDLED_BY]->(x:Response)
RETURN r.id, a.id, h.id, x.id
'''
result = CypherQuery(q).with_config(cfg).execute(datasets)
result.to_pandas()

,r.id,a.id,h.id,x.id
0,R11,aisle-9,oil,slow_down


In [11]:
rows = [tuple(row.values()) for row in result.to_pylist()]
assert rows == [("R11", "aisle-9", "oil", "slow_down")], rows

## Exact answers, but only about stored paths

A graph is precise. Ask about a path that isn't stored and you get nothing back, which is
the right answer. Below, R11 never met the forklift. And when we count visits per aisle,
aisle-7 doesn't show up at all, because no `VISITED` edge points at it.

In [12]:
q = "MATCH (r:Robot {id: 'R11'})-[:VISITED]->(a:Aisle)-[:HAS_HAZARD]->(h:Hazard {id: 'forklift'}) RETURN a.id"
CypherQuery(q).with_config(cfg).execute(datasets).num_rows

0

In [13]:
q = "MATCH (r:Robot)-[:VISITED]->(a:Aisle) RETURN a.id, count(r) AS visits"
CypherQuery(q).with_config(cfg).execute(datasets).to_pandas()

,a.id,visits
0,aisle-8,2
1,aisle-9,2


## Conclusions

In this notebook, we showed how the graph gives exact, auditable answers, but only to exact queries about paths that we
stored, and only when someone (or their agent) writes the right Cypher query.

In the next notebook we build a new kind of
representation using hyperdimensional computing: every node and relationship type becomes a **hypervector**, a list of
10,000 random ±1 numbers. Three simple operators, binding ⊗, bundling ⊕ and permutation ρ,
then let us write an edge such as R04 → aisle-9 as one hypervector:

$$\mathbf{e} = \mathbf{R04} \otimes \mathbf{VISITED} \otimes \rho(\mathbf{aisle9})$$

From hypervectors like this we can **propose** answers via a vector similarity search. The graph we just built helps us
**validate** them.